In [2]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("Chinook_Sqlite.sqlite")

# Create a cursor object
cursor = conn.cursor()


In [3]:
import sqlite3
import os

# --- Database Connection ---
db_file = 'Chinook_Sqlite.sqlite' # Make sure this file is in the same directory or provide the correct path

if not os.path.exists(db_file):
    print(f"Error: Database file '{db_file}' not found.")
    # Exit or handle the error appropriately
    exit()

try:
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    print("Database connected successfully.")
except sqlite3.Error as e:
    print(f"Error connecting to database: {e}")
    exit()

# --- Execution Function ---
def execute_query(query, display_n_rows=5):
    """Execute a SQL query and return the results."""
    #print("\n--- Query ---")
    #print(query)
    #print("\n--- Result ---")
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        if not rows:
            print("No results found.")
        else:
            # Print header (optional, but helpful)
            col_names = [description[0] for description in cursor.description]
            print("=========")
            print(tuple(col_names))
            print("---------")
            # Print rows
            for row in rows[:display_n_rows]:
                print(row)
        print(f"Total rows: {len(rows)}")
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")
    print("="*30)


Database connected successfully.


**Find all customers who’ve ever been billed in a country different from their own.**

In [6]:
query = """
SELECT
  C.CustomerId,
  C.FirstName || ' ' || C.LastName AS CustomerName,
  C.Country
FROM Customer AS C
WHERE EXISTS (
  SELECT 1
  FROM Invoice AS I
  WHERE I.CustomerId = C.CustomerId
    AND I.BillingCountry <> C.Country
);
"""
execute_query(query)

No results found.
Total rows: 0


**List all invoices whose total exceeds the lowest invoice total ever recorded for customers in Canada.**

In [9]:
query = """
SELECT
  InvoiceId,
  CustomerId,
  Total
FROM Invoice
WHERE Total > (
  SELECT MIN(Total)
  FROM Invoice AS I2
  JOIN Customer AS C2 ON I2.CustomerId = C2.CustomerId
  WHERE C2.Country = 'Canada'
);
"""
execute_query(query)

('InvoiceId', 'CustomerId', 'Total')
---------
(1, 2, 1.98)
(2, 4, 3.96)
(3, 8, 5.94)
(4, 14, 8.91)
(5, 23, 13.86)
Total rows: 357


**Find employees who were hired after every “Sales Support Agent” currently on staff.**

In [11]:
query = """
SELECT
  E.EmployeeId,
  E.FirstName || ' ' || E.LastName AS EmployeeName,
  E.Title,
  E.HireDate
FROM Employee AS E
WHERE E.HireDate >  (
  SELECT MAX(HireDate)
  FROM Employee
  WHERE Title = 'Sales Support Agent'
);
"""
execute_query(query)

('EmployeeId', 'EmployeeName', 'Title', 'HireDate')
---------
(7, 'Robert King', 'IT Staff', '2004-01-02 00:00:00')
(8, 'Laura Callahan', 'IT Staff', '2004-03-04 00:00:00')
Total rows: 2


**Top 5 Customers by Total Spending:**

Find the customers who have spent the most money in total across all their invoices.

In [23]:
query = """
WITH CustomerSpending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) AS TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
)
SELECT
    CustomerId,
    FirstName,
    LastName,
    TotalSpent
FROM CustomerSpending
ORDER BY TotalSpent DESC
LIMIT 5;
"""
execute_query(query)

('CustomerId', 'FirstName', 'LastName', 'TotalSpent')
---------
(6, 'Helena', 'Holý', 49.620000000000005)
(26, 'Richard', 'Cunningham', 47.620000000000005)
(57, 'Luis', 'Rojas', 46.62)
(45, 'Ladislav', 'Kovács', 45.62)
(46, 'Hugh', "O'Reilly", 45.62)
Total rows: 5


In [4]:
query = """
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) AS TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId
    
    ORDER BY TotalSpent DESC
    LIMIT 5;
"""
execute_query(query)

('CustomerId', 'FirstName', 'LastName', 'TotalSpent')
---------
(6, 'Helena', 'Holý', 49.620000000000005)
(26, 'Richard', 'Cunningham', 47.620000000000005)
(57, 'Luis', 'Rojas', 46.62)
(45, 'Ladislav', 'Kovács', 45.62)
(46, 'Hugh', "O'Reilly", 45.62)
Total rows: 5


**Sales Agent Performance:**

List each sales support agent and the total sales amount they have generated (based on the customers they support).

In [25]:
query = """
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS AgentName,
    SUM(i.Total) AS TotalSalesGenerated
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
JOIN Invoice i ON c.CustomerId = i.CustomerId
WHERE e.Title = 'Sales Support Agent' 
GROUP BY e.EmployeeId, AgentName
ORDER BY TotalSalesGenerated DESC;
"""
execute_query(query)

('EmployeeId', 'AgentName', 'TotalSalesGenerated')
---------
(3, 'Jane Peacock', 833.0400000000016)
(4, 'Margaret Park', 775.4000000000005)
(5, 'Steve Johnson', 720.1600000000011)
Total rows: 3


In [5]:
query = """
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS AgentName,
    SUM(i.Total) AS TotalSalesGenerated
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
JOIN Invoice i ON c.CustomerId = i.CustomerId
WHERE e.Title = 'Sales Support Agent' 
GROUP BY e.EmployeeId
ORDER BY TotalSalesGenerated DESC;
"""
execute_query(query)

('EmployeeId', 'AgentName', 'TotalSalesGenerated')
---------
(3, 'Jane Peacock', 833.0400000000016)
(4, 'Margaret Park', 775.4000000000005)
(5, 'Steve Johnson', 720.1600000000011)
Total rows: 3


**Most Popular Genre per Country:**

For each country, find the music genre that has the most tracks purchased (based on invoice lines). If there's a tie, list all tied genres.

In [6]:
query = """
WITH CountryGenreSales AS (
    SELECT
        c.Country,
        g.Name AS GenreName,
        COUNT(il.InvoiceLineId) AS TracksSold
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.Name
),
RankedGenreSales AS (
    SELECT
        Country,
        GenreName,
        TracksSold,
        RANK() OVER (PARTITION BY Country ORDER BY TracksSold DESC) AS RankNum
    FROM CountryGenreSales
)
SELECT
    Country,
    GenreName,
    TracksSold
FROM RankedGenreSales
WHERE RankNum = 1
ORDER BY Country, GenreName;
"""
execute_query(query)

('Country', 'GenreName', 'TracksSold')
---------
('Argentina', 'Alternative & Punk', 9)
('Argentina', 'Rock', 9)
('Australia', 'Rock', 22)
('Austria', 'Rock', 15)
('Belgium', 'Rock', 21)
Total rows: 25


In [27]:
query = """
WITH CountryGenreSales AS (
    SELECT
        c.Country AS Country,
        g.Name AS GenreName,
        COUNT(il.InvoiceLineId) AS TracksSold
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.Name
)
SELECT
    Country,
    GenreName,
    TracksSold
FROM (
    SELECT
        Country,
        GenreName,
        TracksSold,
        RANK() OVER (PARTITION BY Country ORDER BY TracksSold DESC) as RankNum
    FROM CountryGenreSales
) ranked
WHERE RankNum = 1
ORDER BY Country, GenreName;
"""
execute_query(query)

('Country', 'GenreName', 'TracksSold')
---------
('Argentina', 'Alternative & Punk', 9)
('Argentina', 'Rock', 9)
('Australia', 'Rock', 22)
('Austria', 'Rock', 15)
('Belgium', 'Rock', 21)
Total rows: 25


In [8]:
query = """
WITH CountryGenreSales AS (
    SELECT
        c.Country,
        g.Name AS GenreName,
        COUNT(il.InvoiceLineId) AS TracksSold,
        RANK() OVER (PARTITION BY Country ORDER BY COUNT(il.InvoiceLineId) DESC) AS RankNum
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.Name
)
SELECT
    Country,
    GenreName,
    TracksSold
FROM CountryGenreSales
WHERE RankNum = 1
ORDER BY Country, GenreName;
"""
execute_query(query)

('Country', 'GenreName', 'TracksSold')
---------
('Argentina', 'Alternative & Punk', 9)
('Argentina', 'Rock', 9)
('Australia', 'Rock', 22)
('Austria', 'Rock', 15)
('Belgium', 'Rock', 21)
Total rows: 25


**Tracks Never Purchased:**

Find all tracks that have never appeared on any invoice line.

In [ ]:
query = """
SELECT
    t.TrackId,
    t.Name AS TrackName,
    a.Title AS AlbumTitle,
    ar.Name AS ArtistName
FROM Track t
JOIN Album a ON t.AlbumId = a.AlbumId
JOIN Artist ar ON a.ArtistId = ar.ArtistId
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE il.InvoiceLineId IS NULL
ORDER BY ArtistName, AlbumTitle, TrackName;
"""
execute_query(query)

('TrackId', 'TrackName', 'AlbumTitle', 'ArtistName')
---------
(11, 'C.O.D.', 'For Those About To Rock We Salute You', 'AC/DC')
(7, "Let's Get It Up", 'For Those About To Rock We Salute You', 'AC/DC')
(18, 'Bad Boy Boogie', 'Let There Be Rock', 'AC/DC')
(17, 'Let There Be Rock', 'Let There Be Rock', 'AC/DC')
(22, 'Whole Lotta Rosie', 'Let There Be Rock', 'AC/DC')
Total rows: 1519


In [10]:
query = """
SELECT
    t.TrackId,
    t.Name AS TrackName
FROM Track t
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE il.InvoiceLineId IS NULL
ORDER BY t.TrackId;
"""
execute_query(query)

('TrackId', 'TrackName')
---------
(7, "Let's Get It Up")
(11, 'C.O.D.')
(17, 'Let There Be Rock')
(18, 'Bad Boy Boogie')
(22, 'Whole Lotta Rosie')
Total rows: 1519


**Customer Spending vs. Average:**

List customers who have spent more than the average total spending across all customers.

In [29]:
query = """
WITH CustomerTotalSpending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) as TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
),
AverageSpending AS (
    SELECT AVG(TotalSpent) as AvgSpent FROM CustomerTotalSpending
)
SELECT
    cts.CustomerId,
    cts.FirstName,
    cts.LastName,
    cts.TotalSpent
FROM CustomerTotalSpending cts
CROSS JOIN AverageSpending avs -- Use CROSS JOIN to get the average available for comparison
WHERE cts.TotalSpent > avs.AvgSpent
ORDER BY cts.TotalSpent DESC;
"""
execute_query(query)

('CustomerId', 'FirstName', 'LastName', 'TotalSpent')
---------
(6, 'Helena', 'Holý', 49.620000000000005)
(26, 'Richard', 'Cunningham', 47.620000000000005)
(57, 'Luis', 'Rojas', 46.62)
(45, 'Ladislav', 'Kovács', 45.62)
(46, 'Hugh', "O'Reilly", 45.62)
Total rows: 22


In [ ]:
query = """
WITH CustomerTotalSpending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(i.Total) as TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
),
AverageSpending AS (
    SELECT AVG(TotalSpent) as AvgSpent FROM CustomerTotalSpending
)
SELECT
    cts.CustomerId,
    cts.FirstName,
    cts.LastName,
    cts.TotalSpent
FROM CustomerTotalSpending cts
CROSS JOIN AverageSpending avs -- Use CROSS JOIN to get the average available for comparison
WHERE cts.TotalSpent > avs.AvgSpent
ORDER BY cts.TotalSpent DESC;
"""
execute_query(query)

**Artist with Most Genres:**

Find the artist who has tracks covering the widest variety of genres.


In [30]:
query = """
WITH ArtistGenreCount AS (
    SELECT
        ar.ArtistId,
        ar.Name AS ArtistName,
        COUNT(DISTINCT g.GenreId) AS DistinctGenreCount
    FROM Artist ar
    JOIN Album al ON ar.ArtistId = al.ArtistId
    JOIN Track t ON al.AlbumId = t.AlbumId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY ar.ArtistId, ar.Name
)
SELECT
    ArtistName,
    DistinctGenreCount
FROM ArtistGenreCount
ORDER BY DistinctGenreCount DESC
LIMIT 1;
"""
execute_query(query)

('ArtistName', 'DistinctGenreCount')
---------
('Iron Maiden', 4)
Total rows: 1


In [20]:
query = """
SELECT
    ar.Name AS ArtistName,
    COUNT(DISTINCT g.GenreId) AS DistinctGenreCount
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track t ON al.AlbumId = t.AlbumId
JOIN Genre g ON t.GenreId = g.GenreId
GROUP BY ar.ArtistId
ORDER BY DistinctGenreCount DESC
LIMIT 1;
"""
execute_query(query)

('ArtistName', 'DistinctGenreCount')
---------
('Iron Maiden', 4)
Total rows: 1


**Percentage of Sales per Genre:**

Calculate the percentage of total revenue generated by each genre.


In [31]:
query = """
WITH GenreSales AS (
    SELECT
        g.Name AS GenreName,
        SUM(il.UnitPrice * il.Quantity) AS GenreRevenue
    FROM Genre g
    JOIN Track t ON g.GenreId = t.GenreId
    JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY g.Name
),
TotalSales AS (
    SELECT SUM(GenreRevenue) AS TotalRevenue FROM GenreSales
)
SELECT
    gs.GenreName,
    gs.GenreRevenue,
    (gs.GenreRevenue * 100.0 / ts.TotalRevenue) AS PercentageOfTotal
FROM GenreSales gs
CROSS JOIN TotalSales ts
ORDER BY PercentageOfTotal DESC;
"""
execute_query(query)

('GenreName', 'GenreRevenue', 'PercentageOfTotal')
---------
('Rock', 826.6500000000061, 35.49987116722505)
('Latin', 382.14000000000203, 16.41071888688484)
('Metal', 261.3600000000009, 11.223911363050751)
('Alternative & Punk', 241.56000000000074, 10.373615047668114)
('TV Shows', 93.52999999999994, 4.016576483724107)
Total rows: 24


**Average Invoice Amount per Country:**

Calculate the average invoice total for each country.


In [ ]:
query = """
SELECT
    c.Country,
    AVG(i.Total) AS AverageInvoiceAmount,
    COUNT(i.InvoiceId) AS NumberOfInvoices
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
HAVING COUNT(i.InvoiceId) > 0 -- Exclude countries with no invoices (if any)
ORDER BY AverageInvoiceAmount DESC;
"""
execute_query(query)



('Country', 'AverageInvoiceAmount', 'NumberOfInvoices')
---------
('Chile', 6.659999999999999, 7)
('Ireland', 6.517142857142857, 7)
('Hungary', 6.517142857142857, 7)
('Czech Republic', 6.445714285714287, 14)
('Austria', 6.088571428571428, 7)
Total rows: 24


**Tracks Appearing in Most Playlists:**

Find the top 10 tracks that appear in the highest number of distinct playlists.

In [ ]:
query = """
SELECT
    t.Name AS TrackName,
    ar.Name AS ArtistName,
    COUNT(DISTINCT pt.PlaylistId) AS PlaylistCount
FROM Track t
JOIN PlaylistTrack pt ON t.TrackId = pt.TrackId
JOIN Album al ON t.AlbumId = al.AlbumId
JOIN Artist ar ON al.ArtistId = ar.ArtistId
GROUP BY t.TrackId, TrackName, ArtistName -- Group by TrackId to count playlists per track
ORDER BY PlaylistCount DESC
LIMIT 10;
"""
execute_query(query)

('TrackName', 'ArtistName', 'PlaylistCount')
---------
('Intoitus: Adorate Deum', 'Alberto Turco & Nova Schola Gregoriana', 5)
('Miserere mei, Deus', 'Richard Marlow & The Choir of Trinity College, Cambridge', 5)
('Aria Mit 30 Veränderungen, BWV 988 "Goldberg Variations": Aria', 'Wilhelm Kempff', 5)
('Suite for Solo Cello No. 1 in G Major, BWV 1007: I. Prélude', 'Yo-Yo Ma', 5)
('The Messiah: Behold, I Tell You a Mystery... The Trumpet Shall Sound', 'Scholars Baroque Ensemble', 5)
Total rows: 10


**Running Total of Monthly Sales:**

Calculate the running total of sales revenue month by month.


In [34]:
query = """
WITH MonthlySales AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS SaleMonth, -- Extract YYYY-MM
        SUM(Total) as MonthlyRevenue
    FROM Invoice
    GROUP BY SaleMonth
)
SELECT
    SaleMonth,
    MonthlyRevenue,
    SUM(MonthlyRevenue) OVER (ORDER BY SaleMonth ASC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RunningTotalRevenue
FROM MonthlySales
ORDER BY SaleMonth;
"""
execute_query(query)

('SaleMonth', 'MonthlyRevenue', 'RunningTotalRevenue')
---------
('2009-01', 35.64, 35.64)
('2009-02', 37.62, 73.25999999999999)
('2009-03', 37.62, 110.88)
('2009-04', 37.62, 148.5)
('2009-05', 37.62, 186.12)
Total rows: 60


**Customer Purchase Frequency:**

Find the average number of days between purchases for each customer who has made more than one purchase.


In [35]:
query = """
WITH CustomerInvoiceDates AS (
    SELECT
        CustomerId,
        InvoiceDate,
        LAG(InvoiceDate, 1) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate) AS PreviousInvoiceDate
    FROM Invoice
),
DaysBetweenPurchases AS (
    SELECT
        CustomerId,
        julianday(InvoiceDate) - julianday(PreviousInvoiceDate) AS DaysDiff -- Calculate difference in days
    FROM CustomerInvoiceDates
    WHERE PreviousInvoiceDate IS NOT NULL -- Only consider rows where a previous purchase exists
)
SELECT
    c.FirstName || ' ' || c.LastName AS CustomerName,
    AVG(dbp.DaysDiff) AS AverageDaysBetweenPurchases
FROM DaysBetweenPurchases dbp
JOIN Customer c ON dbp.CustomerId = c.CustomerId
GROUP BY dbp.CustomerId, CustomerName
HAVING COUNT(dbp.DaysDiff) > 0 -- Ensure we have at least one interval
ORDER BY AverageDaysBetweenPurchases ASC;
"""
execute_query(query)

('CustomerName', 'AverageDaysBetweenPurchases')
---------
('Luís Gonçalves', 207.5)
('František Wichterlová', 207.5)
('Kara Nielsen', 207.5)
('Fernanda Ramos', 207.5)
('Jack Smith', 207.5)
Total rows: 59


**Genre Popularity Trend:**

Show the total number of tracks sold per genre for each year.


In [36]:
query = """
SELECT
    g.Name AS GenreName,
    strftime('%Y', i.InvoiceDate) AS SaleYear,
    COUNT(il.InvoiceLineId) AS TracksSold
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
JOIN Invoice i ON il.InvoiceId = i.InvoiceId
GROUP BY GenreName, SaleYear
ORDER BY GenreName, SaleYear;
"""
execute_query(query)

('GenreName', 'SaleYear', 'TracksSold')
---------
('Alternative', '2010', 6)
('Alternative', '2011', 4)
('Alternative', '2012', 4)
('Alternative & Punk', '2009', 63)
('Alternative & Punk', '2010', 40)
Total rows: 104


**Customers Who Bought Tracks from Multiple Artists of the Same Genre:**

Find customers who have purchased tracks from at least two different artists within the *same* genre (e.g., bought tracks from two different Rock artists).

In [37]:
query = """
WITH CustomerGenreArtist AS (
    SELECT DISTINCT -- Only need unique combinations
        c.CustomerId,
        g.GenreId,
        ar.ArtistId
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    JOIN Album al ON t.AlbumId = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
)
SELECT
    cu.FirstName || ' ' || cu.LastName AS CustomerName,
    g.Name AS GenreName,
    COUNT(cga.ArtistId) AS DifferentArtistsInGenre
FROM CustomerGenreArtist cga
JOIN Customer cu ON cga.CustomerId = cu.CustomerId
JOIN Genre g ON cga.GenreId = g.GenreId
GROUP BY cga.CustomerId, CustomerName, cga.GenreId, GenreName
HAVING COUNT(cga.ArtistId) >= 2 -- The core condition
ORDER BY CustomerName, GenreName;
"""
execute_query(query)

('CustomerName', 'GenreName', 'DifferentArtistsInGenre')
---------
('Aaron Mitchell', 'Latin', 5)
('Aaron Mitchell', 'Metal', 3)
('Aaron Mitchell', 'Rock', 5)
('Alexandre Rocha', 'Alternative & Punk', 3)
('Alexandre Rocha', 'Latin', 5)
Total rows: 220


In [10]:
query = """
    SELECT 
        c.CustomerId,
        c.FirstName || ' ' || c.LastName AS CustomerName,
        g.GenreId,
        g.Name AS GenreName,
        ar.ArtistId,
        COUNT(DISTINCT ar.ArtistId) AS NARTIST
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    JOIN Album al ON t.AlbumId = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
    GROUP BY c.CustomerId,g.GenreId
    HAVING NARTIST>1
    ORDER BY CustomerName, GenreName;
"""
execute_query(query)

('CustomerId', 'CustomerName', 'GenreId', 'GenreName', 'ArtistId', 'NARTIST')
---------
(32, 'Aaron Mitchell', 7, 'Latin', 112, 5)
(32, 'Aaron Mitchell', 3, 'Metal', 106, 3)
(32, 'Aaron Mitchell', 1, 'Rock', 22, 5)
(11, 'Alexandre Rocha', 4, 'Alternative & Punk', 124, 3)
(11, 'Alexandre Rocha', 7, 'Latin', 56, 5)
Total rows: 220


**Longest Track per Genre:**

Find the longest track (in milliseconds) for each genre.


In [ ]:
query = """
WITH TrackRankedByLength AS (
    SELECT
        t.Name AS TrackName,
        g.Name AS GenreName,
        t.Milliseconds,
        RANK() OVER (PARTITION BY g.GenreId ORDER BY t.Milliseconds DESC) as LengthRank
    FROM Track t
    JOIN Genre g ON t.GenreId = g.GenreId
)
SELECT
    GenreName,
    TrackName,
    Milliseconds
FROM TrackRankedByLength
WHERE LengthRank = 1
ORDER BY GenreName;
"""
execute_query(query)


('GenreName', 'TrackName', 'Milliseconds')
---------
('Alternative', 'Reach Down', 672773)
('Alternative & Punk', "Homecoming / The Death Of St. Jimmy / East 12th St. / Nobody Likes You / Rock And Roll Girlfriend / We're Coming Home Again", 558602)
('Blues', "Talkin' 'Bout Women Obviously", 589531)
('Bossa Nova', 'Samba Da Bênção', 409965)
('Classical', 'Adagio for Strings from the String Quartet, Op. 11', 596519)
Total rows: 25


In [23]:
query = """
SELECT
    t.Name AS TrackName,
    g.Name AS GenreName,
    MAX(t.Milliseconds)
FROM Track t
JOIN Genre g ON t.GenreId = g.GenreId
GROUP BY g.GenreId
ORDER BY GenreName;
"""
execute_query(query)


('TrackName', 'GenreName', 'MAX(t.Milliseconds)')
---------
('Reach Down', 'Alternative', 672773)
("Homecoming / The Death Of St. Jimmy / East 12th St. / Nobody Likes You / Rock And Roll Girlfriend / We're Coming Home Again", 'Alternative & Punk', 558602)
("Talkin' 'Bout Women Obviously", 'Blues', 589531)
('Samba Da Bênção', 'Bossa Nova', 409965)
('Adagio for Strings from the String Quartet, Op. 11', 'Classical', 596519)
Total rows: 25


In [12]:
query = """
SELECT
    g.Name AS GenreName,
    t.Name AS TrackName,
    t.Milliseconds
FROM Track t
JOIN Genre g ON t.GenreId = g.GenreId
WHERE t.Milliseconds = (
    SELECT MAX(t2.Milliseconds)
    FROM Track t2
    WHERE t2.GenreId = t.GenreId
)
ORDER BY g.Name;
"""
execute_query(query)

('GenreName', 'TrackName', 'Milliseconds')
---------
('Alternative', 'Reach Down', 672773)
('Alternative & Punk', "Homecoming / The Death Of St. Jimmy / East 12th St. / Nobody Likes You / Rock And Roll Girlfriend / We're Coming Home Again", 558602)
('Blues', "Talkin' 'Bout Women Obviously", 589531)
('Bossa Nova', 'Samba Da Bênção', 409965)
('Classical', 'Adagio for Strings from the String Quartet, Op. 11', 596519)
Total rows: 25


In [13]:
query = """
SELECT
    g.Name AS GenreName,
    t.Name AS TrackName,
    t.Milliseconds
FROM Track t
JOIN Genre g ON t.GenreId = g.GenreId
WHERE t.TrackId = (
    SELECT t2.TrackId
    FROM Track t2
    WHERE t2.GenreId = t.GenreId
    ORDER BY t2.Milliseconds DESC
    LIMIT 1
)
ORDER BY g.Name;
"""
execute_query(query)

('GenreName', 'TrackName', 'Milliseconds')
---------
('Alternative', 'Reach Down', 672773)
('Alternative & Punk', "Homecoming / The Death Of St. Jimmy / East 12th St. / Nobody Likes You / Rock And Roll Girlfriend / We're Coming Home Again", 558602)
('Blues', "Talkin' 'Bout Women Obviously", 589531)
('Bossa Nova', 'Samba Da Bênção', 409965)
('Classical', 'Adagio for Strings from the String Quartet, Op. 11', 596519)
Total rows: 25


**Sales Contribution by Employee's Country:**

Calculate the total sales amount generated by customers grouped by the country of their supporting employee.

In [39]:
query = """
SELECT
    e.Country AS EmployeeCountry,
    SUM(i.Total) AS TotalSales
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY e.Country
ORDER BY TotalSales DESC;
"""
execute_query(query)

('EmployeeCountry', 'TotalSales')
---------
('Canada', 2328.600000000004)
Total rows: 1


**Media Type Usage:**

Show the total number of tracks sold for each media type.


In [40]:
query = """
SELECT
    mt.Name AS MediaTypeName,
    COUNT(il.InvoiceLineId) AS TracksSold
FROM MediaType mt
JOIN Track t ON mt.MediaTypeId = t.MediaTypeId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY mt.MediaTypeId, MediaTypeName
ORDER BY TracksSold DESC;
"""
execute_query(query)

('MediaTypeName', 'TracksSold')
---------
('MPEG audio file', 1976)
('Protected AAC audio file', 146)
('Protected MPEG-4 video file', 111)
('Purchased AAC audio file', 4)
('AAC audio file', 3)
Total rows: 5


In [14]:
query = """
SELECT
    mt.Name AS MediaTypeName,
    COUNT(il.InvoiceLineId) AS TracksSold
FROM MediaType mt
JOIN Track t ON mt.MediaTypeId = t.MediaTypeId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY mt.MediaTypeId
ORDER BY TracksSold DESC;
"""
execute_query(query)

('MediaTypeName', 'TracksSold')
---------
('MPEG audio file', 1976)
('Protected AAC audio file', 146)
('Protected MPEG-4 video file', 111)
('Purchased AAC audio file', 4)
('AAC audio file', 3)
Total rows: 5


**Artists with Only One Album:**

List artists who have exactly one album in the database.

In [41]:
query = """
SELECT
    ar.Name AS ArtistName,
    COUNT(al.AlbumId) AS AlbumCount
FROM Artist ar
LEFT JOIN Album al ON ar.ArtistId = al.ArtistId -- LEFT JOIN to include artists even if they have 0 albums (though schema likely prevents this)
GROUP BY ar.ArtistId, ArtistName
HAVING COUNT(al.AlbumId) = 1
ORDER BY ArtistName;
"""
execute_query(query)

('ArtistName', 'AlbumCount')
---------
('Aaron Copland & London Symphony Orchestra', 1)
('Aaron Goldberg', 1)
('Academy of St. Martin in the Fields & Sir Neville Marriner', 1)
('Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner', 1)
('Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair', 1)
Total rows: 148


In [24]:
query = """
SELECT
    ar.Name AS ArtistName,
    COUNT(al.AlbumId) AS AlbumCount
FROM Artist ar
LEFT JOIN Album al ON ar.ArtistId = al.ArtistId -- LEFT JOIN to include artists even if they have 0 albums (though schema likely prevents this)
GROUP BY ar.ArtistId
HAVING COUNT(al.AlbumId) = 1
ORDER BY ArtistName;
"""
execute_query(query)

('ArtistName', 'AlbumCount')
---------
('Aaron Copland & London Symphony Orchestra', 1)
('Aaron Goldberg', 1)
('Academy of St. Martin in the Fields & Sir Neville Marriner', 1)
('Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner', 1)
('Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair', 1)
Total rows: 148


**Customer Cohort Analysis (First Purchase Month):**

Group customers by the month of their first purchase and calculate the total spending for each cohort.

In [42]:
query = """
WITH FirstPurchase AS (
    SELECT
        CustomerId,
        MIN(InvoiceDate) as FirstInvoiceDate
    FROM Invoice
    GROUP BY CustomerId
),
CustomerCohort AS (
    SELECT
        CustomerId,
        strftime('%Y-%m', FirstInvoiceDate) as CohortMonth
    FROM FirstPurchase
)
SELECT
    cc.CohortMonth,
    SUM(i.Total) AS TotalCohortSpending,
    COUNT(DISTINCT i.CustomerId) AS CohortSize
FROM CustomerCohort cc
JOIN Invoice i ON cc.CustomerId = i.CustomerId
GROUP BY cc.CohortMonth
ORDER BY cc.CohortMonth;
"""
execute_query(query)

('CohortMonth', 'TotalCohortSpending', 'CohortSize')
---------
('2009-01', 233.71999999999994, 6)
('2009-02', 236.7199999999999, 6)
('2009-03', 233.71999999999994, 6)
('2009-04', 196.11999999999998, 5)
('2009-05', 155.47999999999996, 4)
Total rows: 19


In [ ]:
query = """
WITH PURCHASERANKED AS (
    SELECT
        CustomerId,
        InvoiceDate,
        ROW_NUMBER() OVER (PARTITION BY CustomerId ORDER BY strftime("%s",InvoiceDate) ASC) AS DateRank
    FROM Invoice
    GROUP BY CustomerId
),
FirstPurchase AS (
    SELECT
        CustomerId,
        InvoiceDate
    FROM PURCHASERANKED
    WHERE DateRank=1
),
CustomerCohort AS (
    SELECT
        CustomerId,
        strftime('%Y-%m', InvoiceDate) as CohortMonth
    FROM FirstPurchase
)
SELECT
    cc.CohortMonth,
    SUM(i.Total) AS TotalCohortSpending,
    COUNT(DISTINCT i.CustomerId) AS CohortSize
FROM CustomerCohort cc
JOIN Invoice i ON cc.CustomerId = i.CustomerId
GROUP BY cc.CohortMonth
ORDER BY cc.CohortMonth;
"""
execute_query(query)

('CohortMonth', 'TotalCohortSpending', 'CohortSize')
---------
('2009-01', 233.71999999999994, 6)
('2009-02', 236.7199999999999, 6)
('2009-03', 233.71999999999994, 6)
('2009-04', 196.11999999999998, 5)
('2009-05', 155.47999999999996, 4)
Total rows: 19


In [19]:
query = """
WITH FirstPurchase AS (
    SELECT
        CustomerId,
        FIRST_VALUE(InvoiceDate) OVER (PARTITION BY CustomerId ORDER BY strftime("%s",InvoiceDate) ASC) AS InvoiceDate
    FROM Invoice
    GROUP BY CustomerId
),
CustomerCohort AS (
    SELECT
        CustomerId,
        strftime('%Y-%m', InvoiceDate) as CohortMonth
    FROM FirstPurchase
)
SELECT
    cc.CohortMonth,
    SUM(i.Total) AS TotalCohortSpending,
    COUNT(DISTINCT i.CustomerId) AS CohortSize
FROM CustomerCohort cc
JOIN Invoice i ON cc.CustomerId = i.CustomerId
GROUP BY cc.CohortMonth
ORDER BY cc.CohortMonth;
"""
execute_query(query)

('CohortMonth', 'TotalCohortSpending', 'CohortSize')
---------
('2009-01', 233.71999999999994, 6)
('2009-02', 236.7199999999999, 6)
('2009-03', 233.71999999999994, 6)
('2009-04', 196.11999999999998, 5)
('2009-05', 155.47999999999996, 4)
Total rows: 19


**Top 3 Tracks per Playlist:**

For each playlist, list the top 3 longest tracks (in milliseconds).


In [43]:
query = """
WITH PlaylistTrackLengthRank AS (
    SELECT
        p.Name AS PlaylistName,
        t.Name AS TrackName,
        t.Milliseconds,
        ROW_NUMBER() OVER (PARTITION BY p.PlaylistId ORDER BY t.Milliseconds DESC) as RankInPlaylist
    FROM Playlist p
    JOIN PlaylistTrack pt ON p.PlaylistId = pt.PlaylistId
    JOIN Track t ON pt.TrackId = t.TrackId
)
SELECT
    PlaylistName,
    TrackName,
    Milliseconds
FROM PlaylistTrackLengthRank
WHERE RankInPlaylist <= 3
ORDER BY PlaylistName, RankInPlaylist;
"""
execute_query(query)

('PlaylistName', 'TrackName', 'Milliseconds')
---------
('90’s Music', 'Dazed And Confused', 1116734)
('90’s Music', 'Santana Jam', 882834)
('90’s Music', 'The Sun Road', 880640)
('Brazilian Music', 'Vai Passar', 369763)
('Brazilian Music', 'Quanta (Live)', 357485)
Total rows: 38


**Percentage Change in Monthly Sales:**

Calculate the month-over-month percentage change in total sales revenue.


In [44]:
query = """
WITH MonthlySales AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS SaleMonth,
        SUM(Total) as MonthlyRevenue
    FROM Invoice
    GROUP BY SaleMonth
),
LaggedMonthlySales AS (
    SELECT
        SaleMonth,
        MonthlyRevenue,
        LAG(MonthlyRevenue, 1, 0.0) OVER (ORDER BY SaleMonth) AS PreviousMonthRevenue -- Default to 0.0 for first month
    FROM MonthlySales
)
SELECT
    SaleMonth,
    MonthlyRevenue,
    PreviousMonthRevenue,
    CASE
        WHEN PreviousMonthRevenue = 0 OR PreviousMonthRevenue IS NULL THEN NULL -- Avoid division by zero or for the first month
        ELSE ( (MonthlyRevenue - PreviousMonthRevenue) * 100.0 / PreviousMonthRevenue )
    END AS PercentageChange
FROM LaggedMonthlySales
ORDER BY SaleMonth;
"""
execute_query(query)

('SaleMonth', 'MonthlyRevenue', 'PreviousMonthRevenue', 'PercentageChange')
---------
('2009-01', 35.64, 0.0, None)
('2009-02', 37.62, 35.64, 5.5555555555555465)
('2009-03', 37.62, 37.62, 0.0)
('2009-04', 37.62, 37.62, 0.0)
('2009-05', 37.62, 37.62, 0.0)
Total rows: 60


## Some practices of Managing Time

Find all invoices created between 2009-01-01 and 2009-03-31, and list their InvoiceId, CustomerId, InvoiceDate, and Total in ascending order of InvoiceDate.

In [4]:
query = """
SELECT InvoiceId, CustomerId, InvoiceDate, Total
FROM Invoice
WHERE InvoiceDate BETWEEN '2009-01-01' AND '2009-03-31'
ORDER BY InvoiceDate ASC;

"""
execute_query(query)

('InvoiceId', 'CustomerId', 'InvoiceDate', 'Total')
---------
(1, 2, '2009-01-01 00:00:00', 1.98)
(2, 4, '2009-01-02 00:00:00', 3.96)
(3, 8, '2009-01-03 00:00:00', 5.94)
(4, 14, '2009-01-06 00:00:00', 8.91)
(5, 23, '2009-01-11 00:00:00', 13.86)
Total rows: 20


Find all invoice records where the purchase time (the time part of InvoiceDate) was between 00:00:00 and 23:00:00, regardless of the date. Return InvoiceId, CustomerId, InvoiceDate, and BillingCountry.

In [9]:
query = """
SELECT InvoiceId, CustomerId, InvoiceDate, BillingCountry
FROM Invoice
WHERE time(InvoiceDate) BETWEEN '00:00:00' AND '23:00:00'
ORDER BY InvoiceDate ASC;

"""
execute_query(query)


('InvoiceId', 'CustomerId', 'InvoiceDate', 'BillingCountry')
---------
(1, 2, '2009-01-01 00:00:00', 'Germany')
(2, 4, '2009-01-02 00:00:00', 'Norway')
(3, 8, '2009-01-03 00:00:00', 'Belgium')
(4, 14, '2009-01-06 00:00:00', 'Canada')
(5, 23, '2009-01-11 00:00:00', 'USA')
Total rows: 412


For invoices between '2009-01-01' and '2010-12-31', list each InvoiceId, CustomerId, InvoiceDate, Total, and the customer's running total (cumulative sum) of all their purchases so far.

In [10]:
query = """
SELECT
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total,
    SUM(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RunningTotal
FROM Invoice
WHERE InvoiceDate BETWEEN '2009-01-01' AND '2010-12-31'
ORDER BY CustomerId, InvoiceDate;

"""
execute_query(query)


('InvoiceId', 'CustomerId', 'InvoiceDate', 'Total', 'RunningTotal')
---------
(98, 1, '2010-03-11 00:00:00', 3.98, 3.98)
(121, 1, '2010-06-13 00:00:00', 3.96, 7.9399999999999995)
(143, 1, '2010-09-15 00:00:00', 5.94, 13.879999999999999)
(1, 2, '2009-01-01 00:00:00', 1.98, 1.98)
(12, 2, '2009-02-11 00:00:00', 13.86, 15.84)
Total rows: 166


Find all invoices where the purchase time is between '00:00:00' and '16:00:00'. For each invoice, show its InvoiceId, CustomerId, InvoiceDate, Total, and the average invoice total of that customer's invoices made during working hours.



In [13]:
query = """
SELECT
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total,
    AVG(Total) OVER (PARTITION BY CustomerId) AS AvgWorkHourTotal
FROM Invoice
WHERE time(InvoiceDate) BETWEEN '00:00:00' AND '16:00:00'
ORDER BY CustomerId, InvoiceDate;

"""
execute_query(query)


('InvoiceId', 'CustomerId', 'InvoiceDate', 'Total', 'AvgWorkHourTotal')
---------
(98, 1, '2010-03-11 00:00:00', 3.98, 5.659999999999999)
(121, 1, '2010-06-13 00:00:00', 3.96, 5.659999999999999)
(143, 1, '2010-09-15 00:00:00', 5.94, 5.659999999999999)
(195, 1, '2011-05-06 00:00:00', 0.99, 5.659999999999999)
(316, 1, '2012-10-27 00:00:00', 1.98, 5.659999999999999)
Total rows: 412


Let's create a query that analyzes invoice data over date ranges. We'll calculate a running total of sales by month and compare each month's sales to the previous 3 months (a rolling quarter)

In [25]:
query = """
WITH MonthlySales AS (
    SELECT 
        strftime('%Y-%m', InvoiceDate) AS YearMonth,
        SUM(Total) AS MonthTotal
    FROM Invoice
    GROUP BY YearMonth
    ORDER BY YearMonth
)
SELECT 
    YearMonth,
    MonthTotal,
    SUM(MonthTotal) OVER (
        ORDER BY YearMonth 
        RANGE BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS Rolling3MonthTotal,
    AVG(MonthTotal) OVER (
        ORDER BY YearMonth 
        RANGE BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS Rolling3MonthAverage
FROM MonthlySales
ORDER BY YearMonth;
"""
execute_query(query)

('YearMonth', 'MonthTotal', 'Rolling3MonthTotal', 'Rolling3MonthAverage')
---------
('2009-01', 35.64, 35.64, 35.64)
('2009-02', 37.62, 37.62, 37.62)
('2009-03', 37.62, 37.62, 37.62)
('2009-04', 37.62, 37.62, 37.62)
('2009-05', 37.62, 37.62, 37.62)
Total rows: 60


Now let's create a query that analyzes customer purchase patterns within time ranges during the day. We'll determine peak purchase hours and compare each hour's sales to the surrounding time frame.

In [15]:
query = """
WITH HourlySales AS (
    SELECT 
        strftime('%H', InvoiceDate) AS Hour,
        SUM(Total) AS HourTotal,
        COUNT(*) AS TransactionCount
    FROM Invoice
    GROUP BY Hour
    ORDER BY Hour
)
SELECT 
    Hour,
    HourTotal,
    TransactionCount,
    SUM(HourTotal) OVER (
        ORDER BY Hour 
        RANGE BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ) AS Rolling5HourTotal,
    AVG(TransactionCount) OVER (
        ORDER BY Hour 
        RANGE BETWEEN 2 PRECEDING AND 2 FOLLOWING
    ) AS Avg5HourTransactions,
    HourTotal / NULLIF(SUM(HourTotal) OVER (), 0) * 100 AS PercentOfDailyTotal
FROM HourlySales
ORDER BY Hour;
"""
execute_query(query)

('Hour', 'HourTotal', 'TransactionCount', 'Rolling5HourTotal', 'Avg5HourTransactions', 'PercentOfDailyTotal')
---------
('00', 2328.600000000004, 412, 2328.600000000004, 412.0, 100.0)
Total rows: 1


In [16]:
query = """
SELECT
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total,
    SUM(Total) OVER (
        ORDER BY InvoiceDate
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS SumLastTwoAndCurrent
FROM Invoice
ORDER BY InvoiceDate;
"""
execute_query(query)


('InvoiceId', 'CustomerId', 'InvoiceDate', 'Total', 'SumLastTwoAndCurrent')
---------
(1, 2, '2009-01-01 00:00:00', 1.98, 1.98)
(2, 4, '2009-01-02 00:00:00', 3.96, 5.9399999999999995)
(3, 8, '2009-01-03 00:00:00', 5.94, 11.879999999999999)
(4, 14, '2009-01-06 00:00:00', 8.91, 18.81)
(5, 23, '2009-01-11 00:00:00', 13.86, 28.709999999999997)
Total rows: 412


In [18]:
query = """
SELECT i1.InvoiceId, 
       i1.InvoiceDate, 
       i1.Total,
       (
           SELECT SUM(i2.Total)
           FROM Invoice i2
           WHERE i2.InvoiceDate <= i1.InvoiceDate 
             AND i2.InvoiceDate >= datetime(i1.InvoiceDate, '-48 hours')
       ) AS Total_48h
FROM Invoice i1
ORDER BY i1.InvoiceDate;
"""
execute_query(query)

('InvoiceId', 'InvoiceDate', 'Total', 'Total_48h')
---------
(1, '2009-01-01 00:00:00', 1.98, 1.98)
(2, '2009-01-02 00:00:00', 3.96, 5.9399999999999995)
(3, '2009-01-03 00:00:00', 5.94, 11.879999999999999)
(4, '2009-01-06 00:00:00', 8.91, 8.91)
(5, '2009-01-11 00:00:00', 13.86, 13.86)
Total rows: 412
